# Text Embedding Comparison

Goal: find if transcript **embeddings** (from pretrained text encoders) beat or combine well with the 50 manual text/pause/prosodic features from `text_cheating_detection.ipynb`.

**Pipeline shape mirrors `text_cheating_detection.ipynb`:**
- `{folder}GT.csv`, `{folder}_transcripts.json`, `{folder}_features.csv` all live next to the notebook.
- This notebook **reuses** both caches; it does NOT re-transcribe or re-extract manual features. Run `text_cheating_detection.ipynb` first if those caches are missing.

**Experiments (in this order):**
1. Raw embeddings from 3 text models → XGB + RF
2. PCA sweep on each model's embeddings → XGB + RF
3. Manual-feature baselines (all-50 and audio-derived-only)
4. Mix-and-match: replace the text-derived features in the top-50 with raw embedding or PCA(embedding), keep audio-derived features
5. Overall comparison table

**Candidate isolation:** filenames are `{candidate_id}_25.wav` etc. → candidate_id = `filename.rsplit('_', 1)[0]`. Explicit `TEST_FOLDER` removes any overlapping candidates from train; empty `TEST_FOLDER` falls back to `GroupShuffleSplit`.

In [ ]:
# ============================================================
# CONFIG — edit this cell only
# ============================================================
from pathlib import Path

TRAIN_FOLDERS = ["audios2", "audios4"]
TEST_FOLDER   = "audios5"   # empty "" -> 20% candidate-grouped holdout

EMBED_MODELS = [
    ("minilm", "sentence-transformers/all-MiniLM-L6-v2"),
    ("mpnet",  "sentence-transformers/all-mpnet-base-v2"),
    ("bge",    "BAAI/bge-base-en-v1.5"),
]

PCA_COMPONENTS = [16, 32, 64, 128]

# Group labels — match text_cheating_detection's feature groups exactly
TEXT_DERIVED_GROUPS  = ['disfluency', 'stylometric', 'formal_ai', 'perplexity']
AUDIO_DERIVED_GROUPS = ['pause', 'suspicious', 'prosodic', 'voice_q']

TEST_RATIO  = 0.20
RANDOM_SEED = 42

SAVE_DIR = "checkpoints_text_emb"
NB_DIR   = Path(".").resolve()
SAVE_DIR = NB_DIR / SAVE_DIR
SAVE_DIR.mkdir(parents=True, exist_ok=True)

LABEL_MAP = {
    "cheating":1,"read":1,"reading":1,"scripted":1,"yes":1,"1":1,1:1,
    "not cheating":0,"not_cheating":0,"spontaneous":0,"no":0,"0":0,0:0,"genuine":0,
}

print(f"Train: {TRAIN_FOLDERS}  |  Test: {TEST_FOLDER or f'{int(TEST_RATIO*100)}% candidate holdout'}")
print(f"Embedding models: {[t for t,_ in EMBED_MODELS]}")
print(f"PCA components:   {PCA_COMPONENTS}")

In [ ]:
import os, json, gc, warnings
import numpy as np
import pandas as pd
import torch
import joblib
import xgboost as xgb
from tqdm import tqdm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             accuracy_score, confusion_matrix)

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
def candidate_id(filename):
    stem = Path(filename).stem
    parts = stem.rsplit('_', 1)
    return parts[0] if len(parts) == 2 else stem

def scan_folder(name):
    audio_dir = NB_DIR / name
    gt_path   = NB_DIR / f'{name}GT.csv'
    if not gt_path.exists():
        print(f'  SKIP {name}: no {name}GT.csv'); return None
    meta = {
        'name':      name,
        'gt':        gt_path,
        't_json':    NB_DIR / f'{name}_transcripts.json',
        'feat_csv':  NB_DIR / f'{name}_features.csv',
    }
    for tag, _ in EMBED_MODELS:
        meta[f'emb_{tag}'] = NB_DIR / f'{name}_text_emb_{tag}.csv'
    if not meta['t_json'].exists():
        print(f'  WARN {name}: no transcripts ({meta["t_json"].name}). Run text_cheating_detection first.')
    if not meta['feat_csv'].exists():
        print(f'  WARN {name}: no manual features ({meta["feat_csv"].name}).')
    return meta

def load_gt(gt_path):
    gt = pd.read_csv(gt_path)
    fn_col  = next((c for c in gt.columns if c.lower() in ('filename','file','name')), gt.columns[0])
    lbl_col = next((c for c in gt.columns if c.lower() in ('label','class','cheating','gt','label_int','ground_truth')), gt.columns[-1])
    gt = gt.rename(columns={fn_col: 'filename', lbl_col: 'label_raw'})
    gt['label_int'] = gt['label_raw'].map(lambda x: LABEL_MAP.get(x, LABEL_MAP.get(str(x).lower().strip(), -1)))
    gt = gt[gt['label_int'].isin([0,1])][['filename','label_int']].copy()
    gt['label_int'] = gt['label_int'].astype(int)
    return gt

all_names = list(dict.fromkeys(TRAIN_FOLDERS + ([TEST_FOLDER] if TEST_FOLDER else [])))
folders   = [m for m in (scan_folder(n) for n in all_names) if m]

print(f"\n{'folder':<12s} {'trans':>7s}  {'manual':>7s}  " +
      '  '.join(f"{t:>7s}" for t,_ in EMBED_MODELS))
for m in folders:
    def mark(p): return 'cached' if p.exists() else 'need'
    line = f"{m['name']:<12s} {mark(m['t_json']):>7s}  {mark(m['feat_csv']):>7s}"
    for t, _ in EMBED_MODELS:
        line += f"  {mark(m[f'emb_{t}']):>7s}"
    print(line)

## 1. Master Index + Candidate-Isolated Split
Reuses GT + candidate_id like `encoder_comparison.ipynb`.

In [ ]:
rows = []
for m in folders:
    gt = load_gt(m['gt'])
    gt['folder'] = m['name']
    gt['candidate_id'] = gt['filename'].map(candidate_id)
    rows.append(gt)
master = pd.concat(rows, ignore_index=True)
print(f"Master index: {len(master)} rows, {master['candidate_id'].nunique()} candidates")

if TEST_FOLDER:
    test_mask  = master['folder'] == TEST_FOLDER
    test_cands = set(master.loc[test_mask, 'candidate_id'])
    train_mask = (master['folder'].isin([n for n in TRAIN_FOLDERS if n != TEST_FOLDER])
                  & ~master['candidate_id'].isin(test_cands))
    dropped = ((master['folder'].isin([n for n in TRAIN_FOLDERS if n != TEST_FOLDER]))
               & master['candidate_id'].isin(test_cands)).sum()
    print(f"Explicit test folder: {TEST_FOLDER}  |  dropped {int(dropped)} leak rows from train")
else:
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_RATIO, random_state=RANDOM_SEED)
    tr_idx, te_idx = next(gss.split(master, groups=master['candidate_id']))
    train_mask = np.zeros(len(master), dtype=bool); train_mask[tr_idx] = True
    test_mask  = np.zeros(len(master), dtype=bool); test_mask[te_idx]  = True
    print(f"GroupShuffleSplit on candidate_id (test_size={TEST_RATIO})")

train_files = set(master.loc[train_mask, 'filename'])
test_files  = set(master.loc[test_mask,  'filename'])
train_cands = set(master.loc[master['filename'].isin(train_files), 'candidate_id'])
test_cands_final = set(master.loc[master['filename'].isin(test_files), 'candidate_id'])
assert not (train_cands & test_cands_final), 'candidate overlap -- isolation broken'

y_tr_master = master[master['filename'].isin(train_files)][['filename','label_int']]
y_te_master = master[master['filename'].isin(test_files)][['filename','label_int']]
print(f"Train: {len(train_files)} files  ({int((y_tr_master.label_int==1).sum())} cheat / "
      f"{int((y_tr_master.label_int==0).sum())} honest)   {len(train_cands)} candidates")
print(f"Test:  {len(test_files)} files  ({int((y_te_master.label_int==1).sum())} cheat / "
      f"{int((y_te_master.label_int==0).sum())} honest)   {len(test_cands_final)} candidates")

## 2. Train/Eval Helpers + Feature Group Definitions
Feature groups copied from `text_cheating_detection.ipynb` so we can slice the 50-feature CSV by type.

In [ ]:
GROUPS = {
    'disfluency':  ['filler_rate','filler_count','repetition_rate','repair_rate',
                    'discourse_marker_rate','hedge_rate'],
    'stylometric': ['ttr','mattr','mtld','complex_word_rate','avg_word_length',
                    'n_words','n_unique_words','avg_sentence_length','std_sentence_length',
                    'fragment_rate','n_sentences','self_ref_rate',
                    'noun_rate','verb_rate','adj_rate'],
    'pause':       ['pause_mean','pause_std','pause_median','pause_skew','long_pause_rate',
                    'pause_ratio','n_pauses','pause_regularity',
                    'pause_before_content_ratio','pause_before_function_ratio',
                    'mid_phrase_pause_rate','words_per_sec','articulation_rate',
                    'initial_pause','longest_pause'],
    'suspicious':  ['suspicious_gap_count','suspicious_gap_ratio'],
    'formal_ai':   ['formal_transition_count','formal_transition_rate',
                    'ai_phrase_count','ai_phrase_rate'],
    'prosodic':    ['f0_mean','f0_std','f0_range','f0_skew','f0_slope',
                    'energy_mean','energy_std','speaking_rate_std'],
    'voice_q':     ['jitter_local','shimmer_local','hnr_mean'],
    'perplexity':  ['mean_perplexity','burstiness'],
}
TEXT_FEATS  = [f for g in TEXT_DERIVED_GROUPS  for f in GROUPS[g]]
AUDIO_FEATS = [f for g in AUDIO_DERIVED_GROUPS for f in GROUPS[g]]
ALL_FEATS   = TEXT_FEATS + AUDIO_FEATS
print(f"Text-derived:  {len(TEXT_FEATS)}  (groups: {TEXT_DERIVED_GROUPS})")
print(f"Audio-derived: {len(AUDIO_FEATS)} (groups: {AUDIO_DERIVED_GROUPS})")
print(f"All manual:    {len(ALL_FEATS)}")

In [ ]:
PREC_TARGETS = [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

def rec_at_prec_targets(proba, y, targets=PREC_TARGETS, min_tp=3):
    y = np.asarray(y)
    rows = []
    for t in np.arange(0.05, 0.991, 0.005):
        pred = (proba >= t).astype(int)
        tp = int(((pred == 1) & (y == 1)).sum())
        if tp < min_tp: continue
        rows.append((float(t),
                     float(precision_score(y, pred, zero_division=0)),
                     float(recall_score(y, pred, zero_division=0))))
    out = {}
    for tp_target in targets:
        cands = [(thr, p, r) for thr, p, r in rows if p >= tp_target]
        if cands:
            best = max(cands, key=lambda x: x[2])
            out[f'rec@P{int(tp_target*100)}'] = round(best[2], 4)
            out[f'thr@P{int(tp_target*100)}'] = round(best[0], 3)
        else:
            out[f'rec@P{int(tp_target*100)}'] = 0.0
            out[f'thr@P{int(tp_target*100)}'] = None
    return out

def threshold_sweep(proba, y):
    best_thr, best_f1 = 0.5, 0.0
    for thr in np.arange(0.20, 0.81, 0.02):
        f = f1_score(y, (proba >= thr).astype(int), zero_division=0)
        if f > best_f1: best_f1, best_thr = f, thr
    return round(best_thr, 2), round(best_f1, 4)

def eval_one(X_tr, y_tr, X_te, y_te, tag, models=('xgb','rf')):
    sc = StandardScaler().fit(X_tr)
    Xtr, Xte = sc.transform(X_tr), sc.transform(X_te)
    spw = float((y_tr==0).sum()) / max(float((y_tr==1).sum()), 1.0)
    colsample = 0.3 if X_tr.shape[1] > 500 else 0.8
    out = []
    for mdl_name in models:
        if mdl_name == 'xgb':
            clf = xgb.XGBClassifier(
                n_estimators=400, max_depth=5, learning_rate=0.04,
                subsample=0.8, colsample_bytree=colsample, min_child_weight=3,
                scale_pos_weight=spw, eval_metric='logloss',
                early_stopping_rounds=30, random_state=RANDOM_SEED, device='cpu')
            clf.fit(Xtr, y_tr, eval_set=[(Xte, y_te)], verbose=False)
        else:
            clf = RandomForestClassifier(
                n_estimators=500, max_depth=None, min_samples_leaf=2,
                class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)
            clf.fit(Xtr, y_tr)
        proba = clf.predict_proba(Xte)[:, 1]
        thr, f1 = threshold_sweep(proba, y_te)
        pred = (proba >= thr).astype(int)
        cm = confusion_matrix(y_te, pred, labels=[0,1])
        rec_p = rec_at_prec_targets(proba, y_te)
        row = dict(
            tag=f'{tag}_{mdl_name}', n_feat=X_tr.shape[1], thr=thr, f1=f1,
            precision=round(precision_score(y_te, pred, zero_division=0),4),
            recall=round(recall_score(y_te, pred, zero_division=0),4),
            accuracy=round(accuracy_score(y_te, pred),4),
            tp=int(cm[1,1]), fp=int(cm[0,1]), fn=int(cm[1,0]), tn=int(cm[0,0]),
            **rec_p,
        )
        print(f"  {row['tag']:<42s}  F1={row['f1']:.4f}  P={row['precision']:.4f}  R={row['recall']:.4f}  thr={row['thr']}  n_feat={row['n_feat']}")
        print(f"     rec @P60={row['rec@P60']:.3f}  @P65={row['rec@P65']:.3f}  @P70={row['rec@P70']:.3f}  "
              f"@P75={row['rec@P75']:.3f}  @P80={row['rec@P80']:.3f}  @P85={row['rec@P85']:.3f}  @P90={row['rec@P90']:.3f}")
        out.append(row)
    return out

all_results = []
print('Helpers ready.')

## 3. Extract Text Embeddings (per model, cached per folder)
Caches `{folder}_text_emb_{tag}.csv` with columns `filename, te_0 … te_{D-1}`.

In [ ]:
def load_transcripts(meta):
    if not meta['t_json'].exists():
        return {}
    return json.load(open(meta['t_json'], encoding='utf-8'))

def build_text_df(meta):
    t = load_transcripts(meta)
    rows = [{'filename': fn, 'text': (v or {}).get('text','') or ''} for fn, v in t.items()]
    return pd.DataFrame(rows)

def extract_embeddings_for_model(model_tag, model_id):
    need = [m for m in folders if not m[f'emb_{model_tag}'].exists()]
    if not need:
        print(f'  [{model_tag}] all folders cached.'); return
    from sentence_transformers import SentenceTransformer
    print(f"  [{model_tag}] loading {model_id} ...")
    model = SentenceTransformer(model_id, device=DEVICE)
    D = model.get_sentence_embedding_dimension()
    print(f"  [{model_tag}] dim = {D}")
    for m in need:
        df_t = build_text_df(m)
        if df_t.empty:
            print(f"  [{model_tag}] {m['name']}: no transcripts, skipping"); continue
        texts = df_t['text'].tolist()
        print(f"  [{model_tag}] {m['name']}: encoding {len(texts)} texts ...")
        emb = model.encode(texts, batch_size=32, show_progress_bar=True,
                           convert_to_numpy=True, normalize_embeddings=True)
        cols = [f'te_{i}' for i in range(D)]
        out = pd.concat([df_t[['filename']].reset_index(drop=True),
                         pd.DataFrame(emb.astype(np.float32), columns=cols)], axis=1)
        out.to_csv(m[f'emb_{model_tag}'], index=False)
        print(f"  [{model_tag}] saved {m[f'emb_{model_tag}'].name} ({len(out)} rows, {D} dims)")
    del model
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    gc.collect()

def load_emb_cache(model_tag):
    dfs = []
    for m in folders:
        p = m[f'emb_{model_tag}']
        if p.exists(): dfs.append(pd.read_csv(p))
    if not dfs: return None
    return pd.concat(dfs, ignore_index=True)

def split_xy(merged_df, feat_cols):
    tr = merged_df[merged_df['filename'].isin(train_files)]
    te = merged_df[merged_df['filename'].isin(test_files)]
    X_tr = tr[feat_cols].fillna(0).values
    y_tr = tr['label_int'].values
    X_te = te[feat_cols].fillna(0).values
    y_te = te['label_int'].values
    return X_tr, y_tr, X_te, y_te

## 4. Raw Embedding Results (one model at a time)

In [ ]:
emb_frames = {}   # tag -> merged (filename + te_*) + label_int
emb_cols   = {}   # tag -> list of te_ columns

for tag, model_id in EMBED_MODELS:
    print(f"\n=== Raw embeddings: {tag} ({model_id}) ===")
    extract_embeddings_for_model(tag, model_id)
    emb_df = load_emb_cache(tag)
    if emb_df is None:
        print(f"  [{tag}] no embeddings produced, skipping"); continue
    cols = [c for c in emb_df.columns if c.startswith('te_')]
    merged = emb_df.merge(master[['filename','label_int','candidate_id']], on='filename', how='inner')
    emb_frames[tag] = merged
    emb_cols[tag]   = cols
    X_tr, y_tr, X_te, y_te = split_xy(merged, cols)
    print(f"  [{tag}] train={X_tr.shape}  test={X_te.shape}")
    rows = eval_one(X_tr, y_tr, X_te, y_te, tag=f'raw_{tag}')
    all_results.extend(rows)

## 5. PCA Sweep per Model
Fit PCA on TRAIN embeddings only; apply to TEST. Train XGB + RF on reduced features.

In [ ]:
pca_cache = {}   # (tag, k) -> (merged_df_with_pca, pca_cols)

for tag in emb_frames:
    merged = emb_frames[tag]
    cols   = emb_cols[tag]
    tr = merged[merged['filename'].isin(train_files)]
    te = merged[merged['filename'].isin(test_files)]
    X_tr_full = tr[cols].fillna(0).values
    X_te_full = te[cols].fillna(0).values
    max_k = min(PCA_COMPONENTS[-1], X_tr_full.shape[1], X_tr_full.shape[0])
    print(f"\n=== PCA sweep: {tag}  (D={len(cols)} -> up to {max_k})")
    for k in PCA_COMPONENTS:
        if k > max_k:
            print(f"  skip k={k} (> max {max_k})"); continue
        pca = PCA(n_components=k, random_state=RANDOM_SEED).fit(X_tr_full)
        Z_tr = pca.transform(X_tr_full); Z_te = pca.transform(X_te_full)
        # Build merged frame with pca_ cols for reuse in mix-and-match
        pc_cols = [f'pc_{i}' for i in range(k)]
        tr_p = tr[['filename','label_int']].copy(); tr_p[pc_cols] = Z_tr
        te_p = te[['filename','label_int']].copy(); te_p[pc_cols] = Z_te
        frame = pd.concat([tr_p, te_p], ignore_index=True)
        pca_cache[(tag, k)] = (frame, pc_cols, pca)
        rows = eval_one(Z_tr, tr['label_int'].values, Z_te, te['label_int'].values, tag=f'pca{k}_{tag}')
        all_results.extend(rows)

## 6. Manual-Feature Baselines
All-50 manual features + audio-derived-only. Requires `{folder}_features.csv` from `text_cheating_detection.ipynb`.

In [ ]:
manual_frames = []
for m in folders:
    if m['feat_csv'].exists():
        manual_frames.append(pd.read_csv(m['feat_csv']))
manual_df = pd.concat(manual_frames, ignore_index=True) if manual_frames else pd.DataFrame()

if manual_df.empty:
    print("  NO manual features found. Run text_cheating_detection.ipynb first.")
else:
    available_text  = [c for c in TEXT_FEATS  if c in manual_df.columns]
    available_audio = [c for c in AUDIO_FEATS if c in manual_df.columns]
    available_all   = available_text + available_audio
    print(f"Manual features available -> text:{len(available_text)}  audio:{len(available_audio)}  total:{len(available_all)}")
    merged_m = manual_df.merge(master[['filename','label_int']], on='filename', how='inner')

    print("\n=== Manual all ===")
    X_tr, y_tr, X_te, y_te = split_xy(merged_m, available_all)
    all_results.extend(eval_one(X_tr, y_tr, X_te, y_te, tag='manual_all'))

    print("\n=== Manual audio-only ===")
    X_tr, y_tr, X_te, y_te = split_xy(merged_m, available_audio)
    all_results.extend(eval_one(X_tr, y_tr, X_te, y_te, tag='manual_audio'))

    print("\n=== Manual text-only ===")
    X_tr, y_tr, X_te, y_te = split_xy(merged_m, available_text)
    all_results.extend(eval_one(X_tr, y_tr, X_te, y_te, tag='manual_text'))

## 7. Mix-and-Match: replace TEXT features with embedding / PCA(embedding)
Keeps audio-derived features + swaps text-derived features for each embedding (raw) or best-K PCA per model.

In [ ]:
def pick_best_pca_k_for(tag):
    rows = [r for r in all_results if r['tag'].startswith(f'pca') and r['tag'].endswith(f'_{tag}_xgb')]
    if not rows: return None
    best = max(rows, key=lambda r: r['f1'])
    import re
    m = re.match(r'pca(\d+)_', best['tag'])
    return int(m.group(1)) if m else None

if not manual_df.empty:
    # prep audio-only slice keyed by filename
    audio_slice = manual_df[['filename'] + available_audio].copy()

    for tag in emb_frames:
        merged = emb_frames[tag]
        cols   = emb_cols[tag]

        # --- raw embedding + audio-derived manual ---
        mix = merged[['filename','label_int'] + cols].merge(audio_slice, on='filename', how='inner')
        feat_cols = cols + available_audio
        print(f"\n=== audio+raw_{tag} ({len(feat_cols)} feats) ===")
        X_tr, y_tr, X_te, y_te = split_xy(mix, feat_cols)
        all_results.extend(eval_one(X_tr, y_tr, X_te, y_te, tag=f'audio+raw_{tag}'))

        # --- best-K PCA embedding + audio-derived manual ---
        best_k = pick_best_pca_k_for(tag)
        if best_k is not None:
            frame, pc_cols, _ = pca_cache[(tag, best_k)]
            mix = frame[['filename','label_int'] + pc_cols].merge(audio_slice, on='filename', how='inner')
            feat_cols = pc_cols + available_audio
            print(f"\n=== audio+pca{best_k}_{tag} ({len(feat_cols)} feats) ===")
            X_tr, y_tr, X_te, y_te = split_xy(mix, feat_cols)
            all_results.extend(eval_one(X_tr, y_tr, X_te, y_te, tag=f'audio+pca{best_k}_{tag}'))
else:
    print('Skipping mix-and-match: manual features missing.')

## 8. Overall Comparison Table

In [ ]:
cmp_df = pd.DataFrame(all_results).sort_values('f1', ascending=False).reset_index(drop=True)
cmp_df.to_csv(SAVE_DIR / 'text_embedding_comparison.csv', index=False)

print('='*110)
print('  TEXT-EMBEDDING COMPARISON  (sorted by F1)')
print('='*110)
base_cols = ['tag','n_feat','thr','f1','precision','recall','accuracy','tp','fp','fn','tn']
print(cmp_df[[c for c in base_cols if c in cmp_df.columns]].to_string(index=False))
print('='*110)

rp_cols = [f'rec@P{int(p*100)}' for p in PREC_TARGETS if f'rec@P{int(p*100)}' in cmp_df.columns]
if rp_cols:
    print('\n' + '='*110)
    print('  RECALL @ PRECISION TARGETS  (sorted by F1)')
    print('='*110)
    print(cmp_df[['tag','f1','precision','recall'] + rp_cols].to_string(index=False))
    print('='*110)

print(f"\nSaved -> {SAVE_DIR / 'text_embedding_comparison.csv'}")

summary = {
    'train_folders': TRAIN_FOLDERS,
    'test_folder':   TEST_FOLDER,
    'test_ratio':    TEST_RATIO if not TEST_FOLDER else None,
    'embed_models':  dict(EMBED_MODELS),
    'pca_components': PCA_COMPONENTS,
    'text_derived_groups':  TEXT_DERIVED_GROUPS,
    'audio_derived_groups': AUDIO_DERIVED_GROUPS,
    'prec_targets': PREC_TARGETS,
    'n_train': int(len(train_files)),
    'n_test':  int(len(test_files)),
    'results': cmp_df.to_dict(orient='records'),
}
with open(SAVE_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Saved -> {SAVE_DIR / 'summary.json'}")